# B2-019-attention-transformers — Practice p23 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** multi-head-attention, attention-complexity

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The complete index rule is d=head*d_h+head_coordinate, so (n,d)->(floor(d/2),n,d mod 2). Explicitly: for each n=0,1,2, d=0 maps to (0,n,0), d=1 to (0,n,1), d=2 to (1,n,0), and d=3 to (1,n,1). This yields rows [1,2,10,20], [3,4,30,40], [5,6,50,60]. With B=1,h=2,N=3 the score count is 18, within budget 20. With h=4 it is 36, over budget. D determines d_h=D/h, but once h and N are fixed it is absent from B*h*N^2.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 0.0
RTOL = 0.0
H0 = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], dtype=np.float64)
H1 = np.array([[10.0, 20.0], [30.0, 40.0], [50.0, 60.0]], dtype=np.float64)
heads = np.stack((H0, H1), axis=0)
B, N, D, h = 1, 3, 4, 2
BUDGET = 20
concatenated = heads.transpose(1, 0, 2).reshape(N, D)
index_map = {
    (n, d): (d // (D // h), n, d % (D // h))
    for n in range(N)
    for d in range(D)
}
EXPECTED_CONCATENATED = np.array(
    [[1.0, 2.0, 10.0, 20.0],
     [3.0, 4.0, 30.0, 40.0],
     [5.0, 6.0, 50.0, 60.0]],
    dtype=np.float64,
)
score_count_two_heads = B * h * N * N
score_count_four_heads = B * 4 * N * N

### Answer check

In [ ]:
assert len(index_map) == 12
for n in range(N):
    for d in range(D):
        head, row, coordinate = index_map[(n, d)]
        assert concatenated[n, d] == heads[head, row, coordinate]
np.testing.assert_allclose(concatenated, EXPECTED_CONCATENATED, atol=ATOL, rtol=RTOL)
assert score_count_two_heads == 18 <= BUDGET
assert score_count_four_heads == 36 > BUDGET